In [1]:
%pip install pyspark

Note: you may need to restart the kernel to use updated packages.


In [2]:
# 1. Inicialização do PySpark
from pyspark import SparkConf
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

conf = SparkConf()
conf.set('spark.jars.packages', 'org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.11.901')
conf.set('spark.hadoop.fs.s3a.aws.credentials.provider', 'com.amazonaws.auth.InstanceProfileCredentialsProvider')

spark = SparkSession.builder.config(conf=conf).getOrCreate()

:: loading settings :: url = jar:file:/usr/local/lib/python3.7/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-7854d5e3-402c-452a-9822-c8d82050453a;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 521ms :: artifacts dl 14ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.4 from central in [default]
	org.wildfly.openssl#wildfly-openssl;1.0.7.Final from central in [default]
	:: evicted modules:
	com.amazonaws#aws-java-sdk-bundle;1.11.901 by [com.amazonaws#aws-java-sdk-bundle;1.12.262] in [default]
	---------------------------------------------------------------------
	|     

In [3]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

sellers_path = "s3a://last-mile-optimization-trusted/dataset-orders/sellers_cleaned_dataset.csv/"
geolocation_path = "s3a://last-mile-optimization-trusted/dataset-orders/geolocation_cleaned_dataset.csv/"

sellers_df = spark.read.csv(sellers_path, header=True, inferSchema=True)
geolocation_df = spark.read.csv(geolocation_path, header=True, inferSchema=True)

# Create an index per zip for both tables and map sellers to geolocations by modulo
geo_w = Window.partitionBy("geolocation_zip_code_prefix").orderBy(F.monotonically_increasing_id())
geo_indexed = (
    geolocation_df
    .withColumn("geo_index", F.row_number().over(geo_w))
    .withColumn("geo_count", F.count("*").over(Window.partitionBy("geolocation_zip_code_prefix")))
    .withColumnRenamed("geolocation_zip_code_prefix", "seller_zip_code_prefix")
)

geo_counts = geo_indexed.select("seller_zip_code_prefix", "geo_count").dropDuplicates(["seller_zip_code_prefix"])

seller_w = Window.partitionBy("seller_zip_code_prefix").orderBy(F.monotonically_increasing_id())
sellers_indexed = sellers_df.withColumn("seller_index", F.row_number().over(seller_w))

sellers_with_geo_count = sellers_indexed.join(
    geo_counts,
    on="seller_zip_code_prefix",
    how="inner"
)

sellers_with_geo_index = sellers_with_geo_count.withColumn(
    "geo_index",
    F.pmod(F.col("seller_index") - F.lit(1), F.col("geo_count")) + F.lit(1)
)

joined_df = sellers_with_geo_index.join(
    geo_indexed,
    on=["seller_zip_code_prefix", "geo_index"],
    how="inner"
)

joined_df = joined_df.drop("geo_count", "geo_index", "seller_index")

joined_df.show(5)
print(f"Total rows: {joined_df.count()}")

26/04/04 23:09:37 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


+----------------------+--------------------+-----------+-------------------+-------------------+
|seller_zip_code_prefix|           seller_id|seller_city|    geolocation_lat|    geolocation_lng|
+----------------------+--------------------+-----------+-------------------+-------------------+
|                  1021|e0487761face83d64...|  SAO PAULO| -23.54516769470249|-46.632248679583284|
|                  1021|dd55f1bb788714a40...|  SAO PAULO|-23.541026814876787| -46.63207068206479|
|                  1026|c1dde11f12d05c478...|  SAO PAULO|-23.539494799586286|-46.632843914149746|
|                  1026|14a08204d03bb6b6b...|  SAO PAULO| -23.53984025062037|  -46.6302639919499|
|                  1026|c84592044b180dec2...|  SAO PAULO| -23.53984025062037|  -46.6302639919499|
+----------------------+--------------------+-----------+-------------------+-------------------+
only showing top 5 rows



Total rows: 1814


In [4]:
joined_df.coalesce(1) \
    .write \
    .option('header', 'true') \
    .mode('overwrite') \
    .csv('s3a://last-mile-optimization-trusted/dataset-orders/join_sellers_and_geolocation.csv')

spark.stop()

26/04/04 23:10:12 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.
26/04/04 23:10:13 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.
